# 06 — Full Python Data Cleaning & Merge
### Olist Brazilian E-Commerce — one clean, analysis-ready file from all 9 raw CSVs

This notebook is a **single, self-contained, pure-Python/pandas pass** over the raw data — no SQL — that takes all 9 raw Olist CSVs and produces **one clean, merged, order-level CSV** ready for analysis, BI tools, or modeling.

It consolidates and tightens everything learned across Notebooks 1–5:
- All 9 raw files, including `geolocation` (not just the 8 Notebook 1 originally merged)
- A proper **before-cleaning data quality audit**: nulls, duplicates, dtypes, *and* sanity checks on numeric/date logic that Notebook 1 didn't run (negative prices, inverted timestamps, extreme outliers)
- The extra engineered fields Notebooks 2 and 5 showed were actually informative (`freight_ratio`, `approval_hours`, `has_comment`, `comment_length`)
- A documented column dictionary for the final file, so it's usable without re-reading every notebook

**Output:** `olist_master_clean.csv` — one row per order.


In [1]:
import pandas as pd
import numpy as np
import os

pd.set_option('display.max_columns', 60)
pd.set_option('display.width', 160)

DATA_DIR = '../Data'
PROCESSED_DIR = '../Data/processed'
os.makedirs(PROCESSED_DIR, exist_ok=True)


## 1. Load all 9 raw files

In [2]:
orders      = pd.read_csv(f'{DATA_DIR}/olist_orders_dataset.csv')
items       = pd.read_csv(f'{DATA_DIR}/olist_order_items_dataset.csv')
payments    = pd.read_csv(f'{DATA_DIR}/olist_order_payments_dataset.csv')
reviews     = pd.read_csv(f'{DATA_DIR}/olist_order_reviews_dataset.csv')
products    = pd.read_csv(f'{DATA_DIR}/olist_products_dataset.csv')
customers   = pd.read_csv(f'{DATA_DIR}/olist_customers_dataset.csv')
sellers     = pd.read_csv(f'{DATA_DIR}/olist_sellers_dataset.csv')
geoloc      = pd.read_csv(f'{DATA_DIR}/olist_geolocation_dataset.csv')
translation = pd.read_csv(f'{DATA_DIR}/product_category_name_translation.csv')

raw_tables = {
    'orders': orders, 'items': items, 'payments': payments, 'reviews': reviews,
    'products': products, 'customers': customers, 'sellers': sellers,
    'geolocation': geoloc, 'category_translation': translation
}
for name, df in raw_tables.items():
    print(f'{name:22s} shape={df.shape}')


orders                 shape=(99441, 8)
items                  shape=(112650, 7)
payments               shape=(103886, 5)
reviews                shape=(99224, 7)
products               shape=(32951, 9)
customers              shape=(99441, 5)
sellers                shape=(3095, 4)
geolocation            shape=(1000163, 5)
category_translation   shape=(71, 2)


## 2. Data quality audit — before touching anything
Shape, dtypes, nulls, and duplicate primary keys for all 9 tables in one pass.

In [3]:
def audit(name, df, pk=None):
    n_nulls = df.isnull().sum()
    n_nulls = n_nulls[n_nulls > 0]
    dup_pk = df[pk].duplicated().sum() if pk else 'n/a'
    print(f'--- {name} ({df.shape[0]:,} rows, {df.shape[1]} cols) ---')
    print(f'  duplicate {pk or "pk"}: {dup_pk}')
    if len(n_nulls):
        for col, n in n_nulls.items():
            print(f'  null {col:32s} {n:6,} ({n/len(df)*100:.1f}%)')
    else:
        print('  no nulls')
    print()

audit('orders', orders, 'order_id')
audit('items', items)
audit('payments', payments)
audit('reviews', reviews, 'review_id')
audit('products', products, 'product_id')
audit('customers', customers, 'customer_id')
audit('sellers', sellers, 'seller_id')


--- orders (99,441 rows, 8 cols) ---
  duplicate order_id: 0
  null order_approved_at                   160 (0.2%)
  null order_delivered_carrier_date      1,783 (1.8%)
  null order_delivered_customer_date     2,965 (3.0%)

--- items (112,650 rows, 7 cols) ---
  duplicate pk: n/a
  no nulls

--- payments (103,886 rows, 5 cols) ---
  duplicate pk: n/a
  no nulls

--- reviews (99,224 rows, 7 cols) ---
  duplicate review_id: 814
  null review_comment_title             87,656 (88.3%)
  null review_comment_message           58,247 (58.7%)

--- products (32,951 rows, 9 cols) ---
  duplicate product_id: 0
  null product_category_name               610 (1.9%)
  null product_name_lenght                 610 (1.9%)
  null product_description_lenght          610 (1.9%)
  null product_photos_qty                  610 (1.9%)
  null product_weight_g                      2 (0.0%)
  null product_length_cm                     2 (0.0%)
  null product_height_cm                     2 (0.0%)
  null product_w

## 3. Sanity checks on numeric and date logic
Nulls and duplicate keys are the obvious checks — the ones people skip are whether the *values* make sense: negative prices, timestamps that go backwards, and genuinely extreme outliers.

In [4]:
print('price <= 0:                    ', (items['price'] <= 0).sum())
print('freight_value < 0:              ', (items['freight_value'] < 0).sum())

orders_dt = orders.copy()
for c in ['order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date',
          'order_delivered_customer_date', 'order_estimated_delivery_date']:
    orders_dt[c] = pd.to_datetime(orders_dt[c])

delivered_before_purchase = (orders_dt['order_delivered_customer_date'] < orders_dt['order_purchase_timestamp']).sum()
approved_before_purchase = (orders_dt['order_approved_at'] < orders_dt['order_purchase_timestamp']).sum()
print('delivered before purchase (bad):', delivered_before_purchase)
print('approved before purchase (bad): ', approved_before_purchase)

delivery_days = (orders_dt['order_delivered_customer_date'] - orders_dt['order_purchase_timestamp']).dt.total_seconds() / 86400
print()
print('delivery_time_days describe:')
print(delivery_days.describe())
print()
print('orders with delivery time > 60 days:', (delivery_days > 60).sum(), f"({(delivery_days>60).mean()*100:.2f}%)")


price <= 0:                     0
freight_value < 0:               0
delivered before purchase (bad): 0
approved before purchase (bad):  0

delivery_time_days describe:
count    96476.000000
mean        12.558702
std          9.546530
min          0.533414
25%          6.766403
50%         10.217755
75%         15.720327
max        209.628611
dtype: float64

orders with delivery time > 60 days: 306 (0.31%)


**Findings:** no negative prices or freight, and no timestamp inversions (nothing is "delivered" before it was purchased, or "approved" before it was purchased) — the raw data is internally consistent on that front, nothing to fix. There *is* a genuine long tail: 306 orders (0.3%) took over 60 days to deliver, up to a max of 210 days.

**Important judgment call:** a standard statistical outlier rule (1.5×IQR) would flag ~4,900 orders (5%) here — but that's the wrong tool for this dataset, because long delivery times are the actual phenomenon this project is studying, not measurement noise to be scrubbed out. Instead of dropping or capping these rows, we flag them with a boolean column (`is_extreme_delivery`, >60 days) so any downstream analysis or model can choose to exclude them explicitly, without silently losing real signal about Brazil's logistics tail.

## 4. Clean & feature-engineer `orders`

In [5]:
orders_clean = orders.copy()
date_cols = ['order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date',
             'order_delivered_customer_date', 'order_estimated_delivery_date']
for c in date_cols:
    orders_clean[c] = pd.to_datetime(orders_clean[c])

orders_clean['is_delivered'] = (orders_clean['order_status'] == 'delivered').astype(int)

orders_clean['delivery_time_days'] = (
    orders_clean['order_delivered_customer_date'] - orders_clean['order_purchase_timestamp']
).dt.total_seconds() / 86400

orders_clean['delay_days'] = (
    orders_clean['order_delivered_customer_date'] - orders_clean['order_estimated_delivery_date']
).dt.total_seconds() / 86400
orders_clean['is_late'] = (orders_clean['delay_days'] > 0).astype(int)

orders_clean['is_extreme_delivery'] = (orders_clean['delivery_time_days'] > 60).astype(int)

# Payment approval delay (Notebook 5 found this doesn't drive lateness/reviews, but it's still
# a useful operational field to carry forward)
orders_clean['approval_hours'] = (
    orders_clean['order_approved_at'] - orders_clean['order_purchase_timestamp']
).dt.total_seconds() / 3600

print(orders_clean[['is_delivered', 'is_late', 'is_extreme_delivery']].sum())
orders_clean.head(3)


is_delivered           96478
is_late                 7827
is_extreme_delivery      306
dtype: int64


,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,is_delivered,delivery_time_days,delay_days,is_late,is_extreme_delivery,approval_hours
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18,1,8.436574,-7.107488,0,0,0.178333
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13,1,13.782037,-5.355729,0,0,30.713889
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04,1,9.394213,-17.245498,0,0,0.276111


## 5. Clean `products` — fill category gaps, merge English translation

In [6]:
products_clean = products.copy()
products_clean['product_category_name'] = products_clean['product_category_name'].fillna('unknown')
products_clean = products_clean.merge(translation, on='product_category_name', how='left')
products_clean['product_category_name_english'] = products_clean['product_category_name_english'].fillna('unknown')

# Physical dimensions: only 2 rows missing -- leave as NaN rather than inventing a value,
# but flag which rows they are in case a downstream shipping-cost model needs to know
products_clean['missing_dimensions'] = products_clean['product_weight_g'].isnull().astype(int)

print('Categories filled as unknown:', (products['product_category_name'].isnull()).sum())
print('Rows with missing physical dimensions (flagged, not imputed):', products_clean['missing_dimensions'].sum())


Categories filled as unknown: 610
Rows with missing physical dimensions (flagged, not imputed): 2


## 6. Deduplicate `reviews` to one row per order
An order can have more than one review row. Keep the most recent by `review_answer_timestamp`, and carry forward whether it had a written comment and how long it was — Notebook 5 showed this is itself a meaningful signal (bad reviews get a comment 78% of the time vs. 35% for good ones).

In [7]:
reviews_clean = reviews.copy()
reviews_clean['review_creation_date'] = pd.to_datetime(reviews_clean['review_creation_date'])
reviews_clean['review_answer_timestamp'] = pd.to_datetime(reviews_clean['review_answer_timestamp'])

reviews_clean['has_comment'] = (
    reviews_clean['review_comment_message'].notna() &
    (reviews_clean['review_comment_message'].str.strip().str.len() > 0)
).astype(int)
reviews_clean['comment_length'] = reviews_clean['review_comment_message'].fillna('').str.len()

reviews_dedup = (
    reviews_clean.sort_values('review_answer_timestamp')
    .drop_duplicates('order_id', keep='last')
    [['order_id', 'review_score', 'review_creation_date', 'has_comment', 'comment_length']]
)
print(f'Reviews before dedup: {len(reviews_clean):,} -> after dedup: {len(reviews_dedup):,} (one per order)')


Reviews before dedup: 99,224 -> after dedup: 98,673 (one per order)


## 7. Aggregate `order_items` to order grain
One row per order: total item price, freight, item count, freight ratio, and the order's main product category (the category of its single highest-priced item).

In [8]:
items_agg = items.groupby('order_id').agg(
    item_price=('price', 'sum'),
    freight_value=('freight_value', 'sum'),
    n_items=('order_item_id', 'count')
).reset_index()
items_agg['order_revenue'] = items_agg['item_price'] + items_agg['freight_value']
items_agg['freight_ratio'] = items_agg['freight_value'] / items_agg['item_price']

# Main category = category of the highest-priced item in the order
idx_top_item = items.groupby('order_id')['price'].idxmax()
top_items = items.loc[idx_top_item, ['order_id', 'product_id']]
top_items = top_items.merge(products_clean[['product_id', 'product_category_name_english']], on='product_id', how='left')
top_items = top_items.rename(columns={'product_category_name_english': 'main_category'})[['order_id', 'main_category']]

items_agg = items_agg.merge(top_items, on='order_id', how='left')
print(f'{len(items_agg):,} orders with item data (of {len(orders):,} total orders)')
items_agg.head(3)


98,666 orders with item data (of 99,441 total orders)


,order_id,item_price,freight_value,n_items,order_revenue,freight_ratio,main_category
0,00010242fe8c5a6d1ba2dd792cb16214,58.9,13.29,1,72.19,0.225637,cool_stuff
1,00018f77f2f0320c557190d7a144bdd3,239.9,19.93,1,259.83,0.083076,pet_shop
2,000229ec398224ef6ca0657da4fc703e,199.0,17.87,1,216.87,0.089799,furniture_decor


## 8. Aggregate `order_payments` to order grain
An order can be split across multiple payment rows. Roll up to total paid, number of payment rows, max installments, and the dominant payment method (the type behind the largest single payment on the order).

In [9]:
payments_agg = payments.groupby('order_id').agg(
    total_payment=('payment_value', 'sum'),
    n_payment_rows=('payment_sequential', 'count'),
    max_installments=('payment_installments', 'max')
).reset_index()

idx_top_payment = payments.groupby('order_id')['payment_value'].idxmax()
dominant_payment = payments.loc[idx_top_payment, ['order_id', 'payment_type']].rename(
    columns={'payment_type': 'dominant_payment_type'}
)
payments_agg = payments_agg.merge(dominant_payment, on='order_id', how='left')
payments_agg.head(3)


,order_id,total_payment,n_payment_rows,max_installments,dominant_payment_type
0,00010242fe8c5a6d1ba2dd792cb16214,72.19,1,2,credit_card
1,00018f77f2f0320c557190d7a144bdd3,259.83,1,3,credit_card
2,000229ec398224ef6ca0657da4fc703e,216.87,1,5,credit_card


## 9. Aggregate `geolocation` to zip-code grain
Many lat/lng samples exist per zip prefix — average them down to one row per zip so merging onto customers doesn't fan out the row count.

In [10]:
geo_zip = geoloc.groupby('geolocation_zip_code_prefix').agg(
    avg_lat=('geolocation_lat', 'mean'),
    avg_lng=('geolocation_lng', 'mean')
).reset_index().rename(columns={'geolocation_zip_code_prefix': 'zip'})

print(f'{len(geo_zip):,} distinct zip codes with coordinates')


19,015 distinct zip codes with coordinates


## 10. Merge everything into one clean, order-level master table
Every piece above is already at order grain (or zip grain, for the geolocation lookup) — so this merge is a straightforward left-join chain from `orders`, with an assertion at the end confirming no join fanned out the row count.

In [11]:
master = (
    orders_clean
    .merge(customers[['customer_id', 'customer_city', 'customer_state', 'customer_zip_code_prefix']],
           on='customer_id', how='left')
    .merge(geo_zip.rename(columns={'avg_lat': 'customer_lat', 'avg_lng': 'customer_lng'}),
           left_on='customer_zip_code_prefix', right_on='zip', how='left')
    .drop(columns=['zip'])
    .merge(items_agg, on='order_id', how='left')
    .merge(payments_agg, on='order_id', how='left')
    .merge(reviews_dedup, on='order_id', how='left')
)

assert len(master) == len(orders), 'Row count changed during merge -- a join fanned out!'
print(f'Master table: {master.shape[0]:,} rows x {master.shape[1]} columns (matches {len(orders):,} orders exactly)')
master.head(3)


Master table: 99,441 rows x 33 columns (matches 99,441 orders exactly)


,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,is_delivered,delivery_time_days,delay_days,is_late,is_extreme_delivery,approval_hours,customer_city,customer_state,customer_zip_code_prefix,customer_lat,customer_lng,item_price,freight_value,n_items,order_revenue,freight_ratio,main_category,total_payment,n_payment_rows,max_installments,dominant_payment_type,review_score,review_creation_date,has_comment,comment_length
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18,1,8.436574,-7.107488,0,0,0.178333,sao paulo,SP,3149,-23.576983,-46.587161,29.99,8.72,1.0,38.71,0.290764,housewares,38.71,3.0,1.0,voucher,4.0,2017-10-11,1.0,170.0
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13,1,13.782037,-5.355729,0,0,30.713889,barreiras,BA,47813,-12.177924,-44.660711,118.70,22.76,1.0,141.46,0.191744,perfumery,141.46,1.0,1.0,boleto,4.0,2018-08-08,1.0,20.0
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04,1,9.394213,-17.245498,0,0,0.276111,vianopolis,GO,75265,-16.745150,-48.514783,159.90,19.22,1.0,179.12,0.120200,auto,179.12,1.0,3.0,credit_card,5.0,2018-08-18,0.0,0.0


## 11. Post-merge quality check
Confirm coverage of the fields that matter most for analysis, and that dtypes came out sensible.

In [12]:
coverage = pd.DataFrame({
    'non_null': master.notna().sum(),
    'pct_non_null': (master.notna().mean() * 100).round(1)
})
coverage.loc[['order_revenue', 'total_payment', 'review_score', 'main_category', 'customer_lat']]


,non_null,pct_non_null
order_revenue,98666,99.2
total_payment,99440,100.0
review_score,98673,99.2
main_category,98666,99.2
customer_lat,99163,99.7


In [13]:
print('Key numeric summaries:')
master[['delivery_time_days', 'delay_days', 'order_revenue', 'freight_ratio', 'review_score']].describe().round(2)


Key numeric summaries:


,delivery_time_days,delay_days,order_revenue,freight_ratio,review_score
count,96476.00,96476.00,98666.00,98666.00,98673.00
mean,12.56,-11.18,160.58,0.31,4.09
std,9.55,10.19,220.47,0.31,1.35
min,0.53,-146.02,9.59,0.00,1.00
25%,6.77,-16.24,61.98,0.13,4.00
50%,10.22,-11.95,105.29,0.22,5.00
75%,15.72,-6.39,176.87,0.38,5.00
max,209.63,188.98,13664.08,21.45,5.00


## 12. Column dictionary
So the final file is usable without re-reading every notebook.

| Column | Meaning |
|---|---|
| `order_id`, `customer_id` | Primary keys from the raw data |
| `order_status` | Raw order status (`delivered`, `canceled`, `shipped`, ...) |
| `order_purchase_timestamp` ... `order_estimated_delivery_date` | Raw timestamps, cast to datetime |
| `is_delivered` | 1 if `order_status == 'delivered'` |
| `delivery_time_days` | Purchase → customer delivery, in days |
| `delay_days` | Delivery date minus estimated date (positive = late) |
| `is_late` | 1 if `delay_days > 0` |
| `is_extreme_delivery` | 1 if `delivery_time_days > 60` (flag, not a filter) |
| `approval_hours` | Purchase → payment approval, in hours |
| `customer_city`, `customer_state`, `customer_zip_code_prefix` | Customer location |
| `customer_lat`, `customer_lng` | Zip-level average coordinates |
| `item_price`, `freight_value`, `n_items` | Summed/counted from `order_items` |
| `order_revenue` | `item_price + freight_value` |
| `freight_ratio` | `freight_value / item_price` |
| `main_category` | Category of the order's highest-priced item (English) |
| `total_payment`, `n_payment_rows`, `max_installments`, `dominant_payment_type` | From `order_payments` |
| `review_score`, `review_creation_date`, `has_comment`, `comment_length` | Deduplicated, latest review per order |


## 13. Save the final clean, merged file

In [14]:
OUTPUT_PATH = f'{PROCESSED_DIR}/olist_master_clean.csv'
master.to_csv(OUTPUT_PATH, index=False)

print(f'Saved: {OUTPUT_PATH}')
print(f'Shape: {master.shape[0]:,} rows x {master.shape[1]} columns')
print(f'Columns: {list(master.columns)}')


Saved: ../Data/processed/olist_master_clean.csv
Shape: 99,441 rows x 33 columns
Columns: ['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date', 'order_delivered_customer_date', 'order_estimated_delivery_date', 'is_delivered', 'delivery_time_days', 'delay_days', 'is_late', 'is_extreme_delivery', 'approval_hours', 'customer_city', 'customer_state', 'customer_zip_code_prefix', 'customer_lat', 'customer_lng', 'item_price', 'freight_value', 'n_items', 'order_revenue', 'freight_ratio', 'main_category', 'total_payment', 'n_payment_rows', 'max_installments', 'dominant_payment_type', 'review_score', 'review_creation_date', 'has_comment', 'comment_length']


---
**Result:** `olist_master_clean.csv` — one row per order, all 9 raw files merged and cleaned, with the engineered fields that Notebooks 2–5 showed actually matter (lateness, revenue, review behavior, freight ratio, approval delay). This is a single drop-in replacement data source for further Python analysis, a BI dashboard, or a modeling notebook, without needing to touch the raw CSVs or SQL again.
